# Part 5: Pytrees and Model Parameters

**Goal**: Understand JAX's universal data structure — the pytree — and how it enables clean, composable parameter management. By the end, you'll restructure model parameters as proper pytrees, use `jax.tree.map` for gradient updates, and understand how Flax builds on top of this exact abstraction.

---

## Table of Contents

1. **Pytrees — JAX's Universal Container**
2. **Pytrees and Transforms** — `grad` returns a pytree; `tree.map` updates params
3. **Advanced Pytrees** — Custom nodes, filtering/freezing layers
4. **The Bridge to Flax** — Why frameworks exist and what `init`/`apply` mean
5. **Common Misconceptions**
6. **Summary — What To Do Next**

---

> **Prerequisites**: This notebook builds on **Notebook 04: vmap**. The MLP parameters from previous notebooks will be restructured here.

In [1]:
# @title Setup { display-mode: "form" }

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"JAX version: {jax.__version__}")
print(f"Devices:     {jax.devices()}")

JAX version: 0.9.2
Devices:     [CpuDevice(id=0)]


In [2]:
# @title Reconstruct MLP from Notebooks 01-02 { display-mode: "form" }

def init_mlp_params(key, layer_sizes):
    """Initialize MLP parameters with He initialization."""
    params = []
    for i in range(len(layer_sizes) - 1):
        key, w_key, b_key = jax.random.split(key, 3)
        fan_in = layer_sizes[i]
        fan_out = layer_sizes[i + 1]
        w = jax.random.normal(w_key, (fan_in, fan_out)) * jnp.sqrt(2.0 / fan_in)
        b = jnp.zeros(fan_out)
        params.append((w, b))
    return params

def mlp_forward(params, x):
    """Forward pass for a SINGLE input vector."""
    for i, (w, b) in enumerate(params):
        x = x @ w + b
        if i < len(params) - 1:
            x = jnp.maximum(x, 0)
    return x

key = jax.random.PRNGKey(42)
params = init_mlp_params(key, [8, 64, 32, 1])
print("MLP reconstructed: 8 → 64 → 32 → 1")

MLP reconstructed: 8 → 64 → 32 → 1


---

# 4. Pytrees — JAX's Universal Container

So far, our model parameters have been a list of `(weight, bias)` tuples. This works, but manually destructuring `zip(params, grads)` for updates is fragile. JAX has a better abstraction: **pytrees**.

> **A pytree is any nested structure of Python containers (dicts, lists, tuples) with array leaves.**

JAX transforms understand pytrees natively. When you pass a pytree to [`grad`](https://jax.readthedocs.io/en/latest/_autosummary/jax.grad.html), you get back a pytree with the same structure. When you pass a pytree to [`jit`](https://jax.readthedocs.io/en/latest/_autosummary/jax.jit.html), every leaf gets traced.

```
params = {                          grads = {
    'layer0': {                         'layer0': {
        'w': array(...),       grad →       'w': array(...),
        'b': array(...),       ────→        'b': array(...),
    },                                  },
    'layer1': { ... },                  'layer1': { ... },
}                                   }
```

> **What `jax.tree.map(f, tree_a, tree_b)` does**: it walks both pytrees in parallel (they must have the same structure), applies `f(leaf_a, leaf_b)` at every matching pair of leaves, and returns a new pytree with the same structure. Think of it as `zip` for nested structures.
```
<br>

```mermaid
graph TD
    Root[Parameters dict] --> L1[Dense_0 dict]
    Root --> L2[Dense_1 dict]
    
    L1 --> W1("kernel: f32[8, 64]")
    L1 --> B1("bias: f32[64]")
    
    L2 --> W2("kernel: f32[64, 32]")
    L2 --> B2("bias: f32[32]")
    
    classDef leaf fill:#d4edda,stroke:#28a745,stroke-width:2px;
    class W1,B1,W2,B2 leaf
```


In [3]:
# 'Pytree' sounds academic, but it's just fancy JAX jargon for nested Python lists/dicts/tuples.
# We can physically visualize a pytree hierarchy using JAX's tree_util.

print("1. The underlying mathematical shape of our parameters:\n")
print(jax.tree_util.tree_map(lambda arr: arr.shape, params))

print("\n2. The explicit ASCII structure map JAX sees:\n")
print(jax.tree_util.tree_structure(params))

print("\nNotice that a Pytree has 'nodes' (the Python lists and tuples) and 'leaves' (the JAX arrays). ")
print("The magic of JAX is that it can map operations exclusively across the leaves automatically.")


1. The underlying mathematical shape of our parameters:

[((8, 64), (64,)), ((64, 32), (32,)), ((32, 1), (1,))]

2. The explicit ASCII structure map JAX sees:

PyTreeDef([(*, *), (*, *), (*, *)])

Notice that a Pytree has 'nodes' (the Python lists and tuples) and 'leaves' (the JAX arrays). 
The magic of JAX is that it can map operations exclusively across the leaves automatically.


In [4]:
# Any nested structure of dicts/lists/tuples with array leaves is a pytree
example_pytree = {
    'weights': jnp.ones((3, 2)),
    'biases': jnp.zeros(2),
    'config': {
        'scale': jnp.array(0.1),
    }
}

# jax.tree has utilities for working with pytrees
leaves = jax.tree.leaves(example_pytree)
print(f"Number of leaves: {len(leaves)}")
for i, leaf in enumerate(leaves):
    print(f"  Leaf {i}: shape {leaf.shape}, dtype {leaf.dtype}")

# tree structure
structure = jax.tree.structure(example_pytree)
print(f"\nStructure: {structure}")

Number of leaves: 3
  Leaf 0: shape (2,), dtype float32
  Leaf 1: shape (), dtype float32
  Leaf 2: shape (3, 2), dtype float32

Structure: PyTreeDef({'biases': *, 'config': {'scale': *}, 'weights': *})


In [5]:
# tree_map applies a function to every leaf, preserving structure
doubled = jax.tree.map(lambda x: x * 2, example_pytree)

print("Original:")
print(f"  weights[0,:] = {example_pytree['weights'][0]}")
print(f"  scale = {example_pytree['config']['scale']}")

print("\nDoubled:")
print(f"  weights[0,:] = {doubled['weights'][0]}")
print(f"  scale = {doubled['config']['scale']}")

# tree_map with multiple trees (must have matching structure)
a = {'x': jnp.array([1.0, 2.0]), 'y': jnp.array(3.0)}
b = {'x': jnp.array([10.0, 20.0]), 'y': jnp.array(30.0)}

added = jax.tree.map(lambda a, b: a + b, a, b)
print(f"\nAdding two pytrees: {added}")

Original:
  weights[0,:] = [1. 1.]
  scale = 0.10000000149011612

Doubled:
  weights[0,:] = [2. 2.]
  scale = 0.20000000298023224

Adding two pytrees: {'x': Array([11., 22.], dtype=float32), 'y': Array(33., dtype=float32, weak_type=True)}


## Restructuring Our MLP as a Pytree

Let's convert our list-of-tuples parameters to a proper pytree (list of dicts). This makes the code more readable and lets us use `jax.tree.map` for updates.

In [6]:
def init_mlp_pytree(key, layer_sizes):
    """Initialize MLP as a list of {'w': ..., 'b': ...} dicts."""
    params = []
    for i in range(len(layer_sizes) - 1):
        key, w_key, b_key = jax.random.split(key, 3)
        fan_in, fan_out = layer_sizes[i], layer_sizes[i + 1]
        params.append({
            'w': jax.random.normal(w_key, (fan_in, fan_out)) * jnp.sqrt(2.0 / fan_in),
            'b': jnp.zeros(fan_out),
        })
    return params

def mlp_forward_pytree(params, x):
    """Forward pass using pytree params."""
    for i, layer in enumerate(params):
        x = x @ layer['w'] + layer['b']
        if i < len(params) - 1:
            x = jnp.maximum(x, 0)
    return x

key = jax.random.PRNGKey(42)
params_pt = init_mlp_pytree(key, [8, 64, 32, 1])

print("Pytree parameter structure:")
for i, layer in enumerate(params_pt):
    print(f"  Layer {i}: w {layer['w'].shape}, b {layer['b'].shape}")

# Verify: count total parameters
n_params = sum(x.size for x in jax.tree.leaves(params_pt))
print(f"\nTotal parameters: {n_params:,}")

Pytree parameter structure:
  Layer 0: w (8, 64), b (64,)
  Layer 1: w (64, 32), b (32,)
  Layer 2: w (32, 1), b (1,)

Total parameters: 2,689


---

# 5. Pytrees and Transforms

The real power of pytrees: JAX transforms understand them natively. [`grad`](https://jax.readthedocs.io/en/latest/_autosummary/jax.grad.html) through a pytree returns a matching pytree of gradients. `jax.tree.map` over params and grads gives you a clean update rule.

In [7]:
def loss_fn(params, x, y):
    pred = mlp_forward_pytree(params, x)
    return jnp.mean((pred - y) ** 2)

# grad w.r.t. params (first arg) — returns a pytree matching params
x_sample = jax.random.normal(jax.random.PRNGKey(0), (8,))
y_sample = jnp.array(1.0)

grads = jax.grad(loss_fn)(params_pt, x_sample, y_sample)

print("Gradient structure (matches params exactly):")
for i, layer in enumerate(grads):
    print(f"  Layer {i}: dw {layer['w'].shape}, db {layer['b'].shape}")

Gradient structure (matches params exactly):
  Layer 0: dw (8, 64), db (64,)
  Layer 1: dw (64, 32), db (32,)
  Layer 2: dw (32, 1), db (1,)


In [8]:
# SGD update in one line using tree_map
lr = 0.01
updated_params = jax.tree.map(lambda p, g: p - lr * g, params_pt, grads)

# Verify the update happened
print("Before update:")
print(f"  Layer 0 w mean: {jnp.mean(params_pt[0]['w']):.6f}")
print(f"\nAfter update:")
print(f"  Layer 0 w mean: {jnp.mean(updated_params[0]['w']):.6f}")
print(f"\nThis one-liner replaces the manual zip/destructure from Notebook 02.")

Before update:
  Layer 0 w mean: -0.003403

After update:
  Layer 0 w mean: -0.003470

This one-liner replaces the manual zip/destructure from Notebook 02.


## Registering Custom Pytree Nodes (Advanced — Optional)

By default, only dicts, lists, tuples, and NamedTuples are pytree containers. If you want JAX to understand your own class, register it.

> **When would you actually need this?** If you store model parameters in a custom Python class and want to use dot-access (`model.weight`) rather than dict-access (`model['weight']`), or if you're building a library where users shouldn't have to know about pytrees at all. Flax's `nn.Module` is registered this way internally.

In [9]:
@jax.tree_util.register_pytree_node_class
class DenseLayer:
    """A dense layer that JAX knows how to traverse."""

    def __init__(self, w, b):
        self.w = w
        self.b = b

    def tree_flatten(self):
        children = (self.w, self.b)      # The arrays (leaves)
        aux_data = None                   # Non-array data (reconstructed later)
        return children, aux_data

    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(*children)

    def __repr__(self):
        return f"DenseLayer(w={self.w.shape}, b={self.b.shape})"

# Now JAX can traverse it
layer = DenseLayer(jnp.ones((3, 2)), jnp.zeros(2))
print(f"Layer: {layer}")
print(f"Leaves: {jax.tree.leaves(layer)}")

# tree_map works on it
doubled_layer = jax.tree.map(lambda x: x * 2, layer)
print(f"Doubled weights: {doubled_layer.w}")

Layer: DenseLayer(w=(3, 2), b=(2,))
Leaves: [Array([[1., 1.],
       [1., 1.],
       [1., 1.]], dtype=float32), Array([0., 0.], dtype=float32)]
Doubled weights: [[2. 2.]
 [2. 2.]
 [2. 2.]]


## Filtering: Selective Parameter Updates

`jax.tree.map` applies the same function to every leaf. But sometimes you want to treat different parameters differently — for example, freezing certain layers during training.

The trick is to use `jax.tree.map` with a function that checks the leaf and returns zero (or the unmodified param) for layers you want to freeze. Gradients are zeroed out, so the parameter update `param - lr * grad` becomes `param - lr * 0 = param` — the weight never changes.

In [ ]:
# Strategy: create a "freeze mask" pytree with the same structure
# True = freeze (zero out gradient), False = train normally

def make_freeze_mask(params, frozen_layers):
    """Create a boolean mask pytree. True = frozen."""
    mask = []
    for i, layer in enumerate(params):
        frozen = i in frozen_layers
        mask.append(jax.tree.map(lambda _: frozen, layer))
    return mask

# Freeze layer 0
mask = make_freeze_mask(params_pt, frozen_layers={0})
print("Freeze mask:")
for i, layer in enumerate(mask):
    print(f"  Layer {i}: w frozen={layer['w']}, b frozen={layer['b']}")

# Apply: zero out gradients for frozen layers
def masked_update(param, grad, frozen):
    return jnp.where(frozen, param, param - lr * grad)

updated = jax.tree.map(masked_update, params_pt, grads, mask)
print(f"\nLayer 0 w changed? {not jnp.allclose(params_pt[0]['w'], updated[0]['w'])}")
print(f"Layer 1 w changed? {not jnp.allclose(params_pt[1]['w'], updated[1]['w'])}")

Freeze mask:
  Layer 0: w frozen=True, b frozen=True
  Layer 1: w frozen=False, b frozen=False
  Layer 2: w frozen=False, b frozen=False

Layer 0 w changed? False
Layer 1 w changed? True


: 

---

# 4. The Bridge to Flax

You've now seen how to:
- Store parameters as a pytree of nested dicts
- Use `jax.tree.map` to apply gradient updates across every leaf
- Filter or freeze specific parameters by shape or name

This is **exactly** what Flax does under the hood. A Flax model's `params` is a standard pytree — a nested dict of arrays — and `jax.tree.map` is how optimizers update them.

The difference is that Flax automates the tedious parts: inferring layer shapes from dummy inputs, naming parameters consistently, and supporting more complex state (like batch normalization statistics). But the pytree layer is unchanged.

> **Mental model**: `model.init(key, x)` creates the pytree. `model.apply(params, x)` is a pure function that takes the pytree and input and returns output. You still update params with `jax.tree.map` — or with Optax, which does exactly that internally.

In **Notebook 06: Training: Scratch → Flax → Optax**, we'll build a complete training pipeline three ways:
1. From scratch using what you've learned in Notebooks 01–05
2. With Flax `nn.Module` (same params, less boilerplate)
3. With Optax (same update rule, more optimizer options)

---

# 5. Common Misconceptions

## Misconception: "I need Flax or Equinox for model parameters"

For learning JAX? No. Plain dicts and lists are pytrees that JAX understands natively. You can build, train, and deploy models with nothing but [`jax.tree.map`](https://jax.readthedocs.io/en/latest/_autosummary/jax.tree.map.html) and dictionaries. Framework libraries like Flax add convenience (module abstractions, parameter management, state handling) but they are not required — and understanding the pytree layer underneath them makes you a better user of those frameworks.

## Misconception: "All Python objects are pytrees"

Only registered container types (dict, list, tuple, NamedTuple) and their nested combinations are pytrees. Arbitrary Python objects are treated as **leaves** by default — JAX won't look inside them. If you need JAX to traverse a custom class, register it with `register_pytree_node_class` as shown above.

---

# 6. Summary — What To Do Next

## Key Takeaways

1. **A pytree is any nested structure of Python containers with array leaves** — dicts, lists, tuples. JAX transforms understand them natively.

2. **[`jax.tree.map`](https://jax.readthedocs.io/en/latest/_autosummary/jax.tree.map.html) replaces manual destructuring** — `tree_map(lambda p, g: p - lr * g, params, grads)` is cleaner and more general than zip-based loops.

3. **`grad` returns a pytree with the same structure as the params** — you always get a gradient for every leaf you differentiated with respect to.

4. **Custom pytree nodes** extend this to your own classes — Flax, Equinox, and other frameworks use this internally.

5. **Pytrees are the layer on which all JAX ecosystem tools are built** — understanding them makes you a better user of Flax, Optax, and Orbax.

## What's Next

In **Notebook 06: Training: Scratch → Flax → Optax**, we'll build a complete training pipeline and see how Flax and Optax reduce boilerplate while keeping the JAX model identical.

---

# Exercises

1. **Pytree inspection**: Use `jax.tree.leaves()` and `jax.tree.structure()` on a 3-layer MLP params dict. How many leaves are there? What does the treedef look like?

2. **Selective updates**: Given a 4-layer MLP, freeze the first and last layer (zero out their gradients) and train only the middle two layers. Verify the first/last layer weights don't change.

3. **Custom pytree**: Register a `NamedLayer` class (with `.weight` and `.bias` attributes) as a custom pytree node. Verify that `jax.tree.map` traverses it correctly.